In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
import torch
from torch import nn
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from tqdm.notebook import tqdm
from torchvision import transforms, datasets
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
path = "drive/MyDrive/PlantVillage"

In [ ]:
print("Path to dataset files:", path)

In [ ]:
classes = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]

class_name = random.choice(classes)
class_path = os.path.join(path, class_name)

valid_exts = (".jpg", ".jpeg", ".png")
img_files = [f for f in os.listdir(class_path) if f.lower().endswith(valid_exts)]

img_name = random.choice(img_files)
img_path = os.path.join(class_path, img_name)

img = Image.open(img_path)

print(f"Number of total classes: {len(classes)}")
print(class_name)
print(img.height)
print(img.width)

img

In [ ]:
img_as_array = np.asanyarray(img)

plt.figure(figsize=(10,7))
plt.imshow(img_as_array)
plt.axis(False)

print(img_as_array.shape)

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

In [ ]:
def visualize_transform(img_path, transform):
  img = Image.open(img_path)

  transformed_img = transform(img)

  if isinstance(transformed_img, torch.Tensor):
    transformed_img = transforms.ToPILImage()(transformed_img)

  fig, axes = plt.subplots(1, 2, figsize=(8,4))

  axes[0].imshow(img)
  axes[0].set_title("Original")
  axes[0].axis("off")

  axes[1].imshow(transformed_img)
  axes[1].set_title("Transformed")
  axes[1].axis("off")

  plt.show()

In [ ]:
class_name = random.choice(classes)
classs_path = os.path.join(path, class_name)
img_name = random.choice(os.listdir(class_path))
imag_path= os.path.join(class_path, img_name)

visualize_transform(img_path, data_transform)

In [ ]:
full_dataset = datasets.ImageFolder(root=path, transform=None)
full_dataset

In [ ]:
class_to_idx = full_dataset.class_to_idx
classes = list(class_to_idx.keys())
classes

In [ ]:
idx_to_class = {value: key for key, value in class_to_idx.items()}
idx_to_class

In [ ]:
targets = [label for _, label in full_dataset.samples]
print(f"Total images: {len(targets)}")

In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    random_state=42,
    stratify=targets
)

In [ ]:
print(f"Number of Train images: {len(train_idx)} | Number of Test images: {len(test_idx)}")

In [ ]:
calc_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

calc_full = datasets.ImageFolder(root=path, transform=calc_transform)
calc_train = Subset(calc_full, train_idx)

In [ ]:
def get_mean_std(dataset, batch_size=32, num_workers=2):
  loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)
  rgb_sum = torch.zeros(3)
  rgb_sumsq = torch.zeros(3)
  count_pixels = 0

  for x, _ in loader:
    b, c, h, w = x.shape
    count_pixels += b * h * w
    rgb_sum += x.sum(dim=[0,2,3])
    rgb_sumsq += (x**2).sum(dim=[0,2,3])

  mean = rgb_sum / count_pixels
  std = torch.sqrt(rgb_sumsq / count_pixels - mean ** 2)
  return mean, std

In [ ]:
# Mean & std (important code)
mean_tensor, std_tensor = get_mean_std(calc_train)
mean = mean_tensor.tolist()
std = std_tensor.tolist()

print("Mean:", mean)
print("Std:", std)

In [ ]:
mean = [0.45923691987991333, 0.4754456877708435, 0.4114924371242523]
std = [0.18601608276367188, 0.16261300444602966, 0.20084309577941895]

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
train_data_all = datasets.ImageFolder(root=path, transform=train_transform)
test_data_all = datasets.ImageFolder(root=path, transform=test_transform)

train_set = Subset(train_data_all, train_idx)
test_set = Subset(test_data_all, test_idx)

In [ ]:
len(train_set), len(test_set)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_set, batch_size= BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size= BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
len(train_loader), len(test_loader)

In [ ]:

class ImageClassifierCNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.block1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.block2 = nn.Sequential(
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.block3 = nn.Sequential(
        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=128*28*28, out_features=512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(in_features=512, out_features=num_classes)
    )

  def forward(self, x):
    x = self.block1(x)
    x = self.block2(x)
    x = self.block3(x)
    x = self.classifier(x)
    return x
    # return self.classifier(self.block3(self.block2(self.block1(x))))


In [ ]:
print("Using device:", device)


In [ ]:
num_classes = 15

cnn_model = ImageClassifierCNN(num_classes=num_classes).to(device)
cnn_model

In [ ]:
# Create Loss, Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

#Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

#Early Stopping Parameters
patience = 3
early_stop_counter = 0
best_test_loss = float("inf")
best_model_path = "/content/drive/My Drive/cnn_model.pth"


#Move model to GPU if available
cnn_model = cnn_model.to(device)

# Training and Evaluation Loop
EPOCHS = 10

for epoch in range(EPOCHS):

  # Training Phase
  cnn_model.train()
  running_loss, correct, total = 0.0, 0, 0

  # tqdm progress bar
  for batch_idx, (X_batch, y_batch) in enumerate (tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)):
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    #forward pass
    y_pred = cnn_model(X_batch)
    loss = loss_fn(y_pred, y_batch)

    #backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #running metrics
    running_loss += loss.item()
    _, predicted = torch.max(y_pred, 1)
    total += y_batch.size(0)
    correct += (predicted == y_batch).sum().item()

    #batch-wise loss print every 50 batches
    if batch_idx % 50 == 0:
        tqdm.write(f"Batch {batch_idx}: Loss={loss.item():.4f}")

  #Epoch-wise metrics
  train_acc = 100 * correct / total
  avg_loss = running_loss / len(train_loader)

  # Evaluation Phase
  cnn_model.eval()
  test_correct, test_total, test_loss_total = 0, 0, 0.0

  with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            test_pred = cnn_model(X_batch)
            test_loss = loss_fn(test_pred, y_batch)

            test_loss_total += test_loss.item()
            _, predicted = torch.max(test_pred, 1)
            test_total += y_batch.size(0)
            test_correct += (predicted == y_batch).sum().item()

  test_acc = 100 * test_correct / test_total
  avg_loss = test_loss_total / len(test_loader)

  #Learning rate scheduling
  scheduler.step(avg_loss)
  #print epoch summary
  print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | "
        f"Learning rate now: {optimizer.param_groups[0]['lr']}")

  #Early stopping check
  if avg_loss < best_test_loss:
    best_test_loss = avg_loss
    early_stop_counter = 0
    torch.save(cnn_model.state_dict(), best_model_path) #best model save
    print(f"Best model saved at epoch {epoch+1}")
  else:
    early_stop_counter += 1
    print(f"No improvement. Patience counter: {early_stop_counter}/{patience}")
    if early_stop_counter >= patience:
      print("Early stopping triggered!")
      break


In [ ]:
#Load Trained model
num_classes = num_classes
cnn_model = ImageClassifierCNN(num_classes=num_classes)
cnn_model.load_state_dict(torch.load(best_model_path, map_location=device))
cnn_model = cnn_model.to(device)
cnn_model.eval()   # put into inference mode
print("Model loaded and ready for inference.")

In [ ]:
#define same transformations
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
#Load an image for testing
pred_img_path = "/content/drive/MyDrive/PlantVillage/Tomato_Early_blight/0034a551-9512-44e5-ba6c-827f85ecc688___RS_Erly.B 9432.JPG"
pred_img = read_image(pred_img_path).type(torch.float32)
pred_img

In [ ]:
#load image with PIL
pred_img = Image.open(pred_img_path)

#apply transforms
pred_img = test_transform(pred_img)
pred_img = pred_img.unsqueeze(0).to(device)

In [ ]:
#Prediction
cnn_model.eval()
with torch.inference_mode():
  prediction = cnn_model(pred_img)
prediction

In [ ]:
#prediction probabilities using softmax
pred_probabilities = torch.softmax(prediction, dim=1)

#Get the prediction class index
pred_idx = torch.argmax(pred_probabilities, dim=1).item()

#Get the prediction class label
pred_label = idx_to_class[pred_idx]

#print results
print(f"Prediction probabilities:", pred_probabilities.squeeze().tolist())
print(f"Predicted class index:", pred_idx)
print(f"Predicted class label:", pred_label)